In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
import statsmodels.api as sm

# Daten einlesen

In [ ]:
df = pd.read_csv("../data/processed/swissvotes_processed.csv")

# Unterschliedliche Positionen zwiscnen BR und BV

In [ ]:
# Nur Abstimmungen wo BR und Parlament unterschiedlicher Meinung sind
dissens = df[df['br-pos'] != df['bv-pos']].dropna(subset=['br-pos', 'bv-pos'])
print(f"Anzahl Fälle wo BR-Pos != BV-Pos: {len(dissens)}")

In [ ]:

dissens = df[df['br-pos'] != df['bv-pos']].dropna(subset=['br-pos', 'bv-pos'])
print(dissens.groupby('hauptgruppe')['titel_kurz_d'].count().sort_values(ascending=False))

# Modell
Wie stark sollen Ja-Anteil und Stimmbeteiligung in den Zustimmungsfaktor einfliessen?


## Ansatz: Logistische Regression (VERWORFEN)
Zum Bestimmen der Gewichte, fitten wir eine logistische Regression.


In [ ]:
temp = df[['volkja-proz', 'beteiligung', 'rechtsform_name']].dropna()
temp = pd.get_dummies(temp, columns=['rechtsform_name'], drop_first=True)
temp = temp.astype(float)

X = sm.add_constant(temp.drop(columns=['volkja-proz']))
y = temp['volkja-proz']

result = sm.OLS(y, X).fit()
print(result.summary())

Die Regression zeigt, dass der Ja-Anteil kaum durch die Mobilisierung beeinflusst wird. Es gibt keine datengetriebenes Gewichtung. Deshalb müssen wir uns ein anderes Konzept überlegen.

# Kongruenz Rechnung anhand eines ungewichteten Prozentsatzes

In [ ]:
def kongruenz_prozent(df, pos_col, min_n=10):
    """
    Berechnet, wie viel Prozent der Stimmberechtigten
    im Schnitt der Empfehlung eines Akteurs gefolgt sind.

    Parameter:
    - df: DataFrame mit Abstimmungsdaten
    - pos_col: Spaltenname der Position (z.B. 'p-svp')
    - min_n: Mindestanzahl Abstimmungen für gültiges Ergebnis

    Rückgabe:
    - score: Anteil Stimmberechtigte, die im Schnitt folgten (0 bis 1)
    - n: Anzahl ausgewertete Abstimmungen
    """

    # === SCHRITT 1: Daten vorbereiten ===
    # Nur die drei relevanten Spalten behalten
    temp = df[[pos_col, 'volkja-proz', 'beteiligung']].dropna()

    # Sicherstellen, dass die Position eine Zahl ist
    temp[pos_col] = pd.to_numeric(temp[pos_col], errors='coerce')

    # Nur Abstimmungen behalten, wo eine klare Empfehlung vorliegt
    # 1.0 = Ja-Empfehlung, 2.0 = Nein-Empfehlung
    temp = temp[temp[pos_col].isin([1.0, 2.0])]

    # === SCHRITT 2: Genug Daten vorhanden? ===
    if len(temp) < min_n:
        return np.nan, len(temp)

    # === SCHRITT 3: Werte in Anteile umrechnen ===
    # Beispiel: 65% Ja-Stimmen wird zu 0.65
    ja_anteil = temp['volkja-proz'] / 100

    # Beispiel: 45% Beteiligung wird zu 0.45
    beteiligung = temp['beteiligung'] / 100

    # === SCHRITT 4: Wie viele haben gleich gestimmt wie empfohlen? ===
    # Wenn Partei Ja empfiehlt: der Ja-Anteil zählt
    # Wenn Partei Nein empfiehlt: der Nein-Anteil zählt (= 1 minus Ja-Anteil)
    #
    # Beispiel Ja-Empfehlung:  65% stimmten Ja → zustimmung = 0.65
    # Beispiel Nein-Empfehlung: 65% stimmten Ja → zustimmung = 0.35
    ist_ja_empfehlung = (temp[pos_col] == 1.0)
    zustimmung = np.where(ist_ja_empfehlung, ja_anteil, 1 - ja_anteil)

    # === SCHRITT 5: Auf alle Stimmberechtigten umrechnen ===
    # zustimmung bezieht sich nur auf die Stimmenden
    # Wir wollen aber den Anteil ALLER Stimmberechtigten
    #
    # Beispiel: 65% der Stimmenden sagten Ja, Beteiligung 45%
    # → 0.65 * 0.45 = 0.2925 → 29.25% aller Stimmberechtigten folgten
    gefolgt = zustimmung * beteiligung

    # === SCHRITT 6: Durchschnitt über alle Abstimmungen ===
    score = gefolgt.mean()

    return score, len(temp)

In [ ]:
score, n = kongruenz_prozent(df, 'p-svp')
print(f"SVP: {score:.2%} der Stimmberechtigten folgten im Schnitt (n={n})")
score, n = kongruenz_prozent(df, 'p-mitte')
print(f"Mitte: {score:.2%} der Stimmberechtigten folgten im Schnitt (n={n})")
score, n = kongruenz_prozent(df, 'br-pos')
print(f"Bundesrat: {score:.2%} der Stimmberechtigten folgten im Schnitt (n={n})")
score, n = kongruenz_prozent(df, 'p-vcs')
print(f"VCS: {score:.2%} der Stimmberechtigten folgten im Schnitt (n={n})")

## Kongruenz ohne Mobilisierung (Framing nicht mehr Stimmberechtigte sondern in Bezug auf

In [ ]:
df_with_positions = df.copy()
neue_spalten = {}

for col in parteien:
    scores = []

    for i, row in df_with_positions.iterrows():
        position = row[col]
        ja_proz = row['volkja-proz']

        # Keine klare Empfehlung oder keine Daten
        if position not in [1.0, 2.0] or pd.isna(ja_proz):
            scores.append(np.nan)

        # Partei empfiehlt JA → je mehr Ja-Stimmen, desto positiver
        elif position == 1.0:
            scores.append((ja_proz - 50) / 100)

        # Partei empfiehlt NEIN → je mehr Nein-Stimmen, desto positiver
        elif position == 2.0:
            scores.append((50 - ja_proz) / 100)

    neue_spalten[f'zustimmung_{col}'] = scores

df_with_positions = pd.concat(
    [df_with_positions, pd.DataFrame(neue_spalten, index=df_with_positions.index)],
    axis=1
)
print(df_with_positions['zustimmung_br-pos'].dropna().unique())

In [ ]:
df_with_positions.to_csv('../data/processed/df_with_positions.csv', index=False)